# Phase 1 — Data Cleaning
**Ergonomic Risk Factor Prediction for Delivery Riders**

This notebook performs the **data-cleaning** stage of the project pipeline.

| | |
|---|---|
| **Input**  | `data/raw/delivery_rider_survey.csv` — survey responses, 48 columns |
| **Output** | `data/processed/cleaned.csv` — cleaned, analysis-ready dataset |

**Notebook structure**
- **Part A — Inspection** (Steps 1–4): look at the raw data before changing anything.
- **Part B — Cleaning** (Steps 5–9): fix mojibake, trim whitespace, decode Borg emojis,
  convert NASA-TLX to numbers, validate, and save.

Every cleaning decision in Part B is made *after* looking at the real values printed in Part A.

## Variable roles in this dataset

Before cleaning, we document the role each group of columns plays in the study.
This table is **not used by the cleaning code** — it records the project design so a
reviewer can see which variables are predictors and which are outcomes.

| Group | Example columns | Role |
|---|---|---|
| Demographics | Gender, Age, Education, Region, Marital status, Tobacco, Alcohol | Independent (predictor) |
| Work exposure | Platform, Vehicle, Carrying mode, Working hours/day, Days/week, Deliveries/day, Rest break, Job duration, Income | Independent — also the inputs that compute the 6 risk factors |
| NASA-TLX (6 items) | Mental, Physical, Time-pressure, Performance, Effort, Frustration | Independent → `workload_score` |
| Borg CR10 (6 items) | Overall, Legs, Breathing, Lifting, Traffic/weather, Exhaustion | Independent → `fatigue_score`, `force_exertion` |
| Discomfort / MSD | Nordic body-area pain (9), 7-day discomfort (4), leave, doctor, performance-reduced | **Dependent (outcome)** |

> **Key rule:** discomfort is only ever a *dependent / outcome* variable — never a predictor.
> Using it as both an input and an outcome was the methodology flaw corrected in the project plan.

---
# Part A — Inspection

## Step 0 — Setup: imports and file paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

# Resolve the project root whether this notebook runs from notebooks/ or the project root
CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / 'data').exists() else CWD.parent
RAW_CSV   = PROJECT_ROOT / 'data' / 'raw' / 'delivery_rider_survey.csv'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Raw CSV      :', RAW_CSV)
print('Raw CSV found:', RAW_CSV.exists())

## Step 1 — Load the raw data and inspect its structure

In [ ]:
df_raw = pd.read_csv(RAW_CSV)
print(f'Loaded {df_raw.shape[0]} responses x {df_raw.shape[1]} columns')
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
# Missing-value check
missing = df_raw.isna().sum()
missing = missing[missing > 0]
print('Columns containing missing values:', len(missing))
missing if len(missing) else 'No missing values in any column.'

In [ ]:
# Duplicate-row check
print('Fully duplicated rows:', df_raw.duplicated().sum())

## Step 2 — Inspect the categorical columns

We print every distinct value so we can spot mojibake and inconsistent labels.

In [ ]:
for c in df_raw.columns[1:19]:
    print(c)
    print('   ', df_raw[c].value_counts(dropna=False).to_dict())
    print()

## Step 3 — Inspect the NASA-TLX columns

The 6 NASA-TLX workload items are the columns whose names end with `[.]`.
We check how the responses are coded before converting them to a `0–100` scale.

In [ ]:
nasa_cols = [c for c in df_raw.columns if c.endswith('[.]')]
print(f'{len(nasa_cols)} NASA-TLX columns identified:')
for c in nasa_cols:
    print(c)
    print('   ', df_raw[c].value_counts(dropna=False).to_dict())
    print()

## Step 4 — Inspect the Borg CR10 emoji columns

The last 6 columns are an 11-face **Borg CR10** scale stored as emoji. To build a correct
decode map we print each distinct value **with its Unicode code points**.

In [ ]:
borg_cols = list(df_raw.columns[-6:])
all_borg_values = set()
for c in borg_cols:
    print(c)
    for v, n in df_raw[c].value_counts(dropna=False).items():
        all_borg_values.add(v)
        cps = ' '.join(f'U+{ord(ch):04X}' for ch in str(v)) if isinstance(v, str) else str(v)
        print(f'   n={n:<4} value={v!r}  codepoints=[{cps}]')
    print()

print('Distinct values across all 6 Borg columns:', len(all_borg_values))

---
# Part B — Cleaning

Findings from Part A:
- 182 responses, 48 columns — **no missing values, no duplicate rows**.
- No true mojibake; the only non-ASCII text is a curly apostrophe in `Education`.
- NASA-TLX responses are coded `0/25/50/75/100`, with the ends written as text (`00 ( Low )`, `100 ( High )`).
- Borg columns use 11 distinct emoji faces, endpoints labelled `(Extremely Easy)` / `(Extremely Hard)`.

All cleaning steps below modify a working copy, `df`, leaving `df_raw` untouched for comparison.

## Step 5 — Fix mojibake / character encoding

True mojibake would appear as the Unicode replacement character (`U+FFFD`). We scan for it.
The `Education` labels contain a curly apostrophe (`U+2019`) — not corruption, but we
normalise it to a plain apostrophe so the label is consistent and ASCII-safe.

In [ ]:
# Build the working copy that every cleaning step modifies
df = df_raw.copy()

# Scan all text columns for the Unicode replacement character (true mojibake)
mojibake_hits = {}
for c in df.select_dtypes(include='object').columns:
    n = int(df[c].astype(str).str.contains(chr(0xFFFD)).sum())
    if n:
        mojibake_hits[c] = n
print('Columns containing true mojibake (U+FFFD):', mojibake_hits or 'NONE - data is valid UTF-8')

# Normalise the curly apostrophe (U+2019) in Education to a plain apostrophe (U+0027)
print('Education BEFORE:', df['Education'].unique().tolist())
df['Education'] = df['Education'].str.replace(chr(0x2019), chr(0x27), regex=False)
print('Education AFTER :', df['Education'].unique().tolist())

## Step 6 — Trim stray whitespace

Remove leading/trailing whitespace from every column header and every text value, so that
labels group correctly (e.g. `' Yes'` and `'Yes'` are not treated as different categories).

In [ ]:
df.columns = df.columns.str.strip()
text_cols = df.select_dtypes(include='object').columns
for c in text_cols:
    df[c] = df[c].str.strip()
print('Trimmed whitespace from all column headers and', len(text_cols), 'text columns.')

## Step 7 — Decode the Borg CR10 emoji columns

Each of the 6 Borg columns is decoded from an emoji face to an ordinal value `0–10`
(`0` = Extremely Easy, `10` = Extremely Hard). This mapping was confirmed against the
original Google Form:

| Value | Face | Meaning |
|---|---|---|
| 0 | 😁 | Extremely Easy (labelled in data) |
| 1 | 😊 | smiling |
| 2 | 🙂 | slightly smiling |
| 3 | 😐 | neutral |
| 4 | 🙁 | slightly frowning |
| 5 | ☹️ | frowning |
| 6 | 😣 | persevering |
| 7 | 😖 | confounded |
| 8 | 😩 | weary |
| 9 | 😫 | tired |
| 10 | 😭 | Extremely Hard (labelled in data) |

The map is keyed by the **first character** of each cell, so the trailing text labels on
the endpoints (and the variation selector on the frowning face) do not break the lookup.

In [ ]:
# Borg CR10 decode map, keyed by the base emoji code point (0 = easy ... 10 = hard)
borg_face_map = {
    chr(0x1F601): 0,    # Extremely Easy (labelled)
    chr(0x1F60A): 1,    # smiling
    chr(0x1F642): 2,    # slightly smiling
    chr(0x1F610): 3,    # neutral
    chr(0x1F641): 4,    # slightly frowning
    chr(0x2639):  5,    # frowning
    chr(0x1F623): 6,    # persevering
    chr(0x1F616): 7,    # confounded
    chr(0x1F629): 8,    # weary
    chr(0x1F62B): 9,    # tired
    chr(0x1F62D): 10,   # Extremely Hard (labelled)
}

borg_cols = list(df.columns[-6:])
for c in borg_cols:
    decoded = df[c].apply(lambda v: borg_face_map.get(str(v)[0]))
    unmapped = df.loc[decoded.isna(), c].unique()
    assert len(unmapped) == 0, f'Unmapped Borg value in {c!r}: {unmapped}'
    df[c] = decoded.astype('int8')

print('Decoded', len(borg_cols), 'Borg columns to ordinal 0-10:')
for c in borg_cols:
    print('  -', c)
df[borg_cols].describe()

## Step 8 — Convert NASA-TLX responses to a numeric 0–100 scale

The 6 NASA-TLX columns mix plain numbers (`25`, `50`, `75`) with text-tagged endpoints
(`00 ( Low )`, `00 ( low )`, `100 ( High )`). We take the leading number from each value,
which yields a clean `0 / 25 / 50 / 75 / 100` scale and also resolves the `Low`/`low`
capitalisation inconsistency.

In [ ]:
nasa_cols = [c for c in df.columns if c.endswith('[.]')]
assert len(nasa_cols) == 6, f'Expected 6 NASA-TLX columns, found {len(nasa_cols)}'

for c in nasa_cols:
    df[c] = df[c].astype(str).str.strip().str.split().str[0].astype(int)
    unexpected = set(df[c].unique()) - {0, 25, 50, 75, 100}
    assert not unexpected, f'Unexpected NASA-TLX value in {c!r}: {unexpected}'

print('Converted', len(nasa_cols), 'NASA-TLX columns to numeric 0-100.')
df[nasa_cols].describe()

## Step 9 — Final validation and save

We confirm the cleaning did not change the row/column count or introduce missing values,
then save the result to `data/processed/cleaned.csv`.

In [ ]:
assert df.shape == df_raw.shape, 'Row/column count changed during cleaning!'
assert df.isna().sum().sum() == 0, 'Cleaning introduced missing values!'

print('Validation passed:')
print('  Shape unchanged   :', df.shape)
print('  Missing values    :', int(df.isna().sum().sum()))
print('  Borg cols dtype   :', sorted(set(str(t) for t in df[borg_cols].dtypes)))
print('  NASA-TLX cols dtype:', sorted(set(str(t) for t in df[nasa_cols].dtypes)))
df.dtypes.value_counts()

In [ ]:
out_path = PROCESSED / 'cleaned.csv'
df.to_csv(out_path, index=False, encoding='utf-8')
print('Saved cleaned dataset to:', out_path)
print('Final shape:', df.shape[0], 'rows x', df.shape[1], 'columns')
df.head()

## Summary — what Phase 1 did

| Step | Action | Result |
|---|---|---|
| 1–4 | Inspected structure, missing values, duplicates, categories | 182 x 48, 0 missing, 0 duplicates |
| 5 | Mojibake scan + normalise curly apostrophe | No true mojibake; `Education` apostrophe normalised |
| 6 | Trim whitespace from headers and text cells | Consistent labels |
| 7 | Decode 6 Borg CR10 emoji columns | Emoji → ordinal `0–10` |
| 8 | Convert 6 NASA-TLX columns | Text → numeric `0–100` |
| 9 | Validate + save | `data/processed/cleaned.csv` written |

**Next — Phase 2 (Feature engineering):** ordinal-encode the remaining categorical bins,
and derive `workload_score`, `fatigue_score`, `force_exertion`, `vehicle_rank`, and the
`discomfort` target. Column renaming to short names also happens in Phase 2.